<a href="https://colab.research.google.com/github/ritambrakumari/building-llm/blob/main/tokeni.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**STEP 1 TOKENIZATION**

In [10]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    rawtext = f.read()

print(rawtext[:500])  # preview

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it'


In [11]:
import re #for splitting we use re library in pyhon
text="hello world. this is a test"
result=re.split(r'(\s)',text) #split the text whenever whitespace is encountered
__builtins__.print(result) # Use __builtins__.print to avoid conflict

['hello', ' ', 'world.', ' ', 'this', ' ', 'is', ' ', 'a', ' ', 'test']


In [12]:
#splitting comma and period
result=re.split(r'([.,]|\s)',text)
__builtins__.print(result)

['hello', ' ', 'world', '.', '', ' ', 'this', ' ', 'is', ' ', 'a', ' ', 'test']


In [13]:
#removing the redudant whitespace

result=[item for item in result if item.strip()]
__builtins__.print(result)

['hello', 'world', '.', 'this', 'is', 'a', 'test']


In [14]:
import re
# Use the same regex as in the tokenizer and process rawtext
preprocessed = re.split(r'([,.:;?_!"()])|--|\s', rawtext)
preprocessed = [
    item.strip()for item in preprocessed if item is not None and item.strip()
]

In [15]:
print(len(preprocessed))


4422


**STEP 2 CREATING TOKEN IDS**

Token ids are usually assigned alphabetically and uniquely

In [16]:
all_words=sorted(set(preprocessed))
vocab_size=len(all_words)  #contains unique word
print(vocab_size)


1153


In [17]:
vocab= {token:integer for integer, token in enumerate(all_words)} #ennumerates command generally assign number in python

In [18]:
vocab= {token:integer for integer, token in enumerate(all_words)} #ennumerates command generally assign number in python

In [19]:
for i, item in enumerate(vocab.items()):#punctuations are given priority
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
("'Are", 3)
("'It's", 4)
("'coming'", 5)
("'done'", 6)
("'subject", 7)
("'technique'", 8)
("'way", 9)
('(', 10)
(')', 11)
(',', 12)
('.', 13)
(':', 14)
(';', 15)
('?', 16)
('A', 17)
('Ah', 18)
('Among', 19)
('And', 20)
('Arrt', 21)
('As', 22)
('At', 23)
('Be', 24)
('Begin', 25)
('Burlington', 26)
('But', 27)
('By', 28)
('Carlo', 29)
('Chicago', 30)
('Claude', 31)
('Come', 32)
('Croft', 33)
('Destroyed', 34)
('Devonshire', 35)
("Don't", 36)
('Dubarry', 37)
('Emperors', 38)
('Florence', 39)
('For', 40)
('Gallery', 41)
('Gideon', 42)
('Gisburn', 43)
("Gisburn's", 44)
('Gisburns', 45)
('Grafton', 46)
('Greek', 47)
('Grindle', 48)
("Grindle's", 49)
('Grindles', 50)


In [20]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab #token to token id
        self.int_to_str = {i:s for s,i in vocab.items()} #converting token id into token

    def encode(self, text):
        # Use the same regex as vocabulary building, and convert tokens to lowercase
        preprocessed = re.split(r'([,.:;?_!"()])|--|\s', text)
        preprocessed = [
            item.strip()for item in preprocessed if item is not None and item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        print(preprocessed)
        return ids

    def decode(self, ids):
      text = " ".join([self.int_to_str[i] for i in ids])
      text = re.sub(r'\s+([,.:;?!"()])', r'\1', text)
      return text

In [21]:
missing = [token for token in preprocessed if token not in vocab]
print(missing)

[]


Encoding the text


In [22]:
tokenizer=SimpleTokenizerV1(vocab)

text = """It's the last he painted, you know,"
            Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

["It's", 'the', 'last', 'he', 'painted', ',', 'you', 'know', ',', '"', 'Gisburn', 'said', 'with', 'pardonable', 'pride', '.']
[68, 1006, 623, 549, 767, 12, 1147, 617, 12, 1, 43, 870, 1129, 775, 814, 13]


decoding the text


In [23]:
tokenizer.decode(ids)

'It\'s the last he painted, you know," Gisburn said with pardonable pride.'

Dealing with special context token
special context token is used to deal whenever there is no words in tokenizer



In [24]:
#Adding 2 tokens unk and end of text
all_token=sorted(list(set(preprocessed)))
all_token.extend(["<|endoftext|>","<|unk|>"]) #extend add additional token
vocab={token:integer for integer, token in enumerate(all_token)}

In [25]:
len(vocab.items())

1155

In [26]:
for i, item in enumerate(list(vocab.items())[-5:]): #last 5 entries of updated vocabulary
    print(item)


('younger', 1150)
('your', 1151)
('yourself', 1152)
('<|endoftext|>', 1153)
('<|unk|>', 1154)


In [27]:

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [28]:
tokenizer=SimpleTokenizerV2(vocab)
text1="Hello do you like tea?"
text2="in the sunlight terraces of palace"

text="<|endoftext|>".join((text1,text2))
print(text)

Hello do you like tea?<|endoftext|>in the sunlight terraces of palace


In [29]:
tokenizer.encode(text)

[1154, 371, 1147, 650, 994, 16, 1154, 1006, 1154, 1002, 743, 1154]

In [30]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|> do you like tea? <|unk|> the <|unk|> terraces of <|unk|>'

BYTE PAIR ENCODING


In [31]:
!pip3 install tiktoken

In [32]:
import importlib
import tiktoken
print("tiktoken version", importlib.metadata.version("tiktoken"))

tiktoken version 0.13.0


In [33]:
#instantiating the BPE tokenizer from tiktoken
tokenizer=tiktoken.get_encoding("gpt2")

In [34]:
#converting into Token ids "ENCODING"
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [35]:
#converting token ids into "DECODING"
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [36]:
#How BPE tokenizer deals with unkmown token
integers = tokenizer.encode("Akwirw ier")
print(integers)

strings = tokenizer.decode(integers)
print(strings)

[33901, 86, 343, 86, 220, 959]
Akwirw ier
